<a href="https://colab.research.google.com/github/Aaditya-git/Aaditya-git/blob/main/HW3_DM_TASK_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
DATA_ZIP = "kmeans_data.zip"
OUTDIR = "task1_outputs"
MAX_ITER = 500
RANDOM_STATE = 42


In [ ]:
import os, zipfile, time, json, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs(OUTDIR, exist_ok=True)

# Unzip dataset
assert os.path.exists(DATA_ZIP), f"Zip not found at: {DATA_ZIP}"
with zipfile.ZipFile(DATA_ZIP, 'r') as z:
    z.extractall("kmeans_data")

# Expect data.csv and label.csv inside the zip
data_path = "kmeans_data/data.csv"
label_path = "kmeans_data/label.csv"
assert os.path.exists(data_path) and os.path.exists(label_path), "Expected data.csv and label.csv in the zip."

# Load data
X = pd.read_csv(data_path, header=None).to_numpy(dtype=float)   # shape: (10000, 784)
y = pd.read_csv(label_path, header=None).iloc[:,0].to_numpy()   # shape: (10000,)
n, d = X.shape
classes = np.unique(y)
K = len(classes)
print(f"Loaded X: {X.shape}, y: {y.shape}, K={K}, classes={classes}")


Loaded X: (10000, 784), y: (10000,), K=10, classes=[0 1 2 3 4 5 6 7 8 9]


In [ ]:
def euclidean(a, b):
    diff = a - b
    return np.sqrt(np.dot(diff, diff))

def cosine_dissim(a, b):
    # 1 - cosine similarity
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0.0:
        return 1.0
    return 1.0 - (np.dot(a, b) / denom)

def generalized_jaccard_dissim(a, b):
    mins = np.minimum(a, b)
    maxs = np.maximum(a, b)
    denom = np.sum(maxs)
    if denom == 0.0:
        return 0.0
    return 1.0 - (np.sum(mins) / denom)


In [ ]:
# ===========================
#  K-Means Variants (from scratch)
# ===========================
def sse_like(X, labels, centroids, metric):
    total = 0.0
    if metric == "euclidean":
        for i, x in enumerate(X):
            d = euclidean(x, centroids[labels[i]])
            total += d*d
    elif metric == "cosine":
        for i, x in enumerate(X):
            d = cosine_dissim(x, centroids[labels[i]])
            total += d*d
    elif metric == "jaccard":
        for i, x in enumerate(X):
            d = generalized_jaccard_dissim(x, centroids[labels[i]])
            total += d*d
    else:
        raise ValueError("unknown metric")
    return float(total)

def pairwise_assign(X, centroids, metric):
    n = X.shape[0]; k = centroids.shape[0]
    labels = np.empty(n, dtype=int)
    if metric == "euclidean":
        for i in range(n):
            dmin, argmin = float("inf"), -1
            for j in range(k):
                d = euclidean(X[i], centroids[j])
                if d < dmin:
                    dmin, argmin = d, j
            labels[i] = argmin
    elif metric == "cosine":
        for i in range(n):
            dmin, argmin = float("inf"), -1
            for j in range(k):
                d = cosine_dissim(X[i], centroids[j])
                if d < dmin:
                    dmin, argmin = d, j
            labels[i] = argmin
    elif metric == "jaccard":
        for i in range(n):
            dmin, argmin = float("inf"), -1
            for j in range(k):
                d = generalized_jaccard_dissim(X[i], centroids[j])
                if d < dmin:
                    dmin, argmin = d, j
            labels[i] = argmin
    else:
        raise ValueError("unknown metric")
    return labels

def recompute_centroids(X, labels, k, metric):
    d = X.shape[1]
    C = np.zeros((k, d), dtype=float)
    counts = np.zeros(k, dtype=int)
    for i, x in enumerate(X):
        c = labels[i]
        C[c] += x
        counts[c] += 1
    for j in range(k):
        if counts[j] > 0:
            C[j] /= counts[j]
        else:
            C[j] = X[np.random.randint(0, X.shape[0])]
        if metric == "cosine":
            nrm = np.linalg.norm(C[j])
            if nrm > 0:
                C[j] = C[j] / nrm
    return C

def kmeans_fit(X, k, metric="euclidean", max_iter=500, random_state=42, stop_on_sse_increase=True):
    rng = np.random.default_rng(random_state)
    n = X.shape[0]

    idx = rng.integers(0, n)
    centroids = [X[idx].copy()]
    for _ in range(1, k):
        d2 = np.array([min(np.sum((x - c)**2) for c in centroids) for x in X], dtype=float)
        probs = d2 / d2.sum() if d2.sum() > 0 else np.ones_like(d2)/len(d2)
        idx = rng.choice(n, p=probs)
        centroids.append(X[idx].copy())
    centroids = np.vstack(centroids)

    if metric == "cosine":
        for j in range(k):
            nrm = np.linalg.norm(centroids[j])
            if nrm > 0:
                centroids[j] = centroids[j] / nrm

    prev_sse = float("inf")
    history = []
    start = time.perf_counter()

    for it in range(1, max_iter+1):
        labels = pairwise_assign(X, centroids, metric)
        newC = recompute_centroids(X, labels, k, metric)
        sse = sse_like(X, labels, newC, metric)
        wall = time.perf_counter() - start
        history.append({"iter": it, "sse": float(sse), "time": float(wall)})

        no_change = np.allclose(newC, centroids, atol=1e-12, rtol=0.0)
        sse_increased = (sse > prev_sse) if prev_sse != float("inf") else False

        centroids, prev_sse = newC, sse

        if no_change or (stop_on_sse_increase and sse_increased) or it == max_iter:
            return {"centroids": centroids, "labels": labels, "iterations": it, "history": history, "final_sse": sse}

    return {"centroids": centroids, "labels": labels, "iterations": max_iter, "history": history, "final_sse": prev_sse}

from collections import Counter

def majority_vote_map(y_true, cluster_assignments, k):
    mapping = {}
    for c in range(k):
        idxs = np.where(cluster_assignments == c)[0]
        if len(idxs) == 0:
            mapping[c] = None
        else:
            votes = Counter(y_true[idxs])
            mapping[c] = votes.most_common(1)[0][0]
    return mapping

def clustering_accuracy(y_true, cluster_assignments, k):
    mapping = majority_vote_map(y_true, cluster_assignments, k)
    correct = 0
    for i, c in enumerate(cluster_assignments):
        lab = mapping.get(c, None)
        if lab is not None and y_true[i] == lab:
            correct += 1
    return correct / len(y_true), mapping


In [ ]:
# (A) Euclidean:
X_eu = X.copy()

# (B) Cosine: L2-normalized per sample
norms = np.linalg.norm(X, axis=1, keepdims=True)
X_cos = X / np.maximum(norms, 1e-12)

# (C) Jaccard: min-max to [0,1] per feature to guarantee nonnegativity
X_min = X.min(axis=0, keepdims=True)
X_max = X.max(axis=0, keepdims=True)
den = np.maximum(X_max - X_min, 1e-12)
X_jac = (X - X_min) / den

X_sets = {"euclidean": X_eu, "cosine": X_cos, "jaccard": X_jac}


In [ ]:
# ===========================
# Run Experiments (3 stop rules)
# ===========================
def run_condition(mode_label):
    results = []
    histories = {}
    if mode_label == "no_change":
        stop_flag = False
    elif mode_label == "sse_increase":
        stop_flag = True
    elif mode_label == "max_iter":

        # it can still stop on 'no_change' if it truly converges earlier -> we'll report actual iters. :))))
        stop_flag = False
    else:
        raise ValueError("Unknown mode")

    for metric in ["euclidean", "cosine", "jaccard"]:
        out = kmeans_fit(
            X_sets[metric], K, metric=metric,
            max_iter=MAX_ITER, random_state=RANDOM_STATE,
            stop_on_sse_increase=stop_flag
        )
        acc, lblmap = clustering_accuracy(y, out["labels"], K)
        results.append({
            "metric": metric,
            "iterations": out["iterations"],
            "sse": out["final_sse"],
            "acc": acc,
            "label_map": lblmap
        })
        histories[metric] = out["history"]
    return pd.DataFrame(results), histories

def save_hist_plots(histories, title_prefix):
    for metric, hist in histories.items():
        it = [h["iter"] for h in hist]
        sse = [h["sse"] for h in hist]
        plt.figure()
        plt.plot(it, sse, marker="o")
        plt.xlabel("Iteration")
        plt.ylabel("SSE-like objective")
        plt.title(f"{title_prefix}: {metric}")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTDIR, f"{title_prefix}_{metric}_sse.png"))
        plt.close()


df_sse_inc, H_sse_inc = run_condition("sse_increase")
df_no_change, H_no_change = run_condition("no_change")
df_max_iter, H_max_iter = run_condition("max_iter")

p1 = os.path.join(OUTDIR, "summary_sse_increase.csv"); df_sse_inc.to_csv(p1, index=False)
p2 = os.path.join(OUTDIR, "summary_no_change.csv");   df_no_change.to_csv(p2, index=False)
p3 = os.path.join(OUTDIR, "summary_max_iter.csv");    df_max_iter.to_csv(p3, index=False)

save_hist_plots(H_sse_inc, "sse_increase_history")
save_hist_plots(H_no_change, "no_change_history")
save_hist_plots(H_max_iter, "max_iter_history")

print("Saved:")
print(p1); print(p2); print(p3)


Saved:
task1_outputs/summary_sse_increase.csv
task1_outputs/summary_no_change.csv
task1_outputs/summary_max_iter.csv


In [ ]:
import numpy as np
import json
import os

best_by_sse = df_sse_inc.loc[df_sse_inc["sse"].idxmin()].to_dict()
best_by_acc = df_sse_inc.loc[df_sse_inc["acc"].idxmax()].to_dict()

# Convert NumPy objects to Python-native types for JSON serialization ,, for json object errorr!! :))))
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    else:
        return obj

summary_best = {
    "best_by_sse": make_serializable(best_by_sse),
    "best_by_accuracy": make_serializable(best_by_acc),
}

with open(os.path.join(OUTDIR, "best_under_sse_increase.json"), "w") as f:
    json.dump(summary_best, f, indent=2)

df_sse_inc, df_no_change, df_max_iter, summary_best


(      metric  iterations           sse     acc  \
 0  euclidean          32  2.532360e+10  0.6018   
 1     cosine          43  6.838989e+02  0.5795   
 2    jaccard          11  3.684989e+03  0.5888   
 
                                            label_map  
 0  {0: 2, 1: 3, 2: 7, 3: 1, 4: 8, 5: 5, 6: 0, 7: ...  
 1  {0: 3, 1: 0, 2: 2, 3: 7, 4: 1, 5: 5, 6: 1, 7: ...  
 2  {0: 2, 1: 1, 2: 0, 3: 1, 4: 6, 5: 8, 6: 3, 7: ...  ,
       metric  iterations           sse     acc  \
 0  euclidean          32  2.532360e+10  0.6018   
 1     cosine          87  6.825054e+02  0.6159   
 2    jaccard         106  3.660795e+03  0.6028   
 
                                            label_map  
 0  {0: 2, 1: 3, 2: 7, 3: 1, 4: 8, 5: 5, 6: 0, 7: ...  
 1  {0: 3, 1: 0, 2: 2, 3: 7, 4: 1, 5: 8, 6: 1, 7: ...  
 2  {0: 2, 1: 1, 2: 0, 3: 1, 4: 6, 5: 8, 6: 3, 7: ...  ,
       metric  iterations           sse     acc  \
 0  euclidean          32  2.532360e+10  0.6018   
 1     cosine          87  6.825054e

In [ ]:
# ===========================
# Quick table for Q4 (SSEs across termination conditions)
# ===========================
merged = (df_no_change[["metric","sse"]].rename(columns={"sse":"no_change_sse"})
          .merge(df_sse_inc[["metric","sse"]].rename(columns={"sse":"sse_increase_sse"}), on="metric")
          .merge(df_max_iter[["metric","sse"]].rename(columns={"sse":"max_iter_sse"}), on="metric"))
merged


,metric,no_change_sse,sse_increase_sse,max_iter_sse
0,euclidean,2.532360e+10,2.532360e+10,2.532360e+10
1,cosine,6.825054e+02,6.838989e+02,6.825054e+02
2,jaccard,3.660795e+03,3.684989e+03,3.660795e+03


In [ ]:
from IPython.display import display

cols = ["metric", "iterations", "sse", "acc"]
display(df_sse_inc[cols])


,metric,iterations,sse,acc
0,euclidean,32,2.532360e+10,0.6018
1,cosine,43,6.838989e+02,0.5795
2,jaccard,11,3.684989e+03,0.5888
